# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

Scraping process:
1) Go through each page and find the ones that have “member”
2) Go into each details section of each person who has “member” in their bio
3) Calculate how many there are from the top 100 (first 5 pages of 20 people) + give the proportion
4) Then get the average number of dramas they're in

In [8]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
import re
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

options = Options()
options.add_argument("--headless")

driver = webdriver.Firefox(options=options)


In [9]:
driver.get("https://mydramalist.com/people/top")
print(driver.title)

Top Actors - MyDramaList


In [3]:
#with Driver(browser="Firefox") as driver:

In [14]:
# Go through the first 5 pages and search for "member"
# in <meta property="og:description" content="">
# Save the name/link <meta property="og:title" content=""> of each one as idol

PROFILE_RE = re.compile(r"/people/\d+-")
MEMBER_RE = re.compile(r"\bmembers?\b", re.IGNORECASE)

wait = WebDriverWait(driver, 10)
idol = []
seen = set()

for page in range(1, 6):
    driver.get(f"https://mydramalist.com/people/top?page={page}")

    try:
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, 'a[href*="/people/"]')
        ))
    except TimeoutException:
        print(f"Page {page}: timed out (title: {driver.title!r})")
        continue

    # Collect profile links first to avoid stale elements
    links = []
    for a in driver.find_elements(By.CSS_SELECTOR, 'a[href*="/people/"]'):
        href = a.get_attribute("href")
        if href and PROFILE_RE.search(href) and href not in seen:
            seen.add(href)
            links.append(href)

    print(f"Page {page}: {len(links)} profiles")

    # Visit each profile
    for link in links:
        try:
            driver.get(link)

            desc = driver.find_element(
                By.CSS_SELECTOR, 'meta[property="og:description"]'
            ).get_attribute("content") or ""

            if MEMBER_RE.search(desc):
                title = driver.find_element(
                    By.CSS_SELECTOR, 'meta[property="og:title"]'
                ).get_attribute("content") or ""
                name = re.sub(r"\s*-\s*MyDramaList\s*$", "", title)
                idol.append((name, link))

        except NoSuchElementException:
            print(f"Missing meta tag: {link}")
        except Exception as e:
            print(f"Error on {link}: {e}")

        time.sleep(1)

print(len(idol), "matches")
for name, link in idol:
    print(name, link)


Page 1: 20 profiles
Page 2: 20 profiles
Page 3: 20 profiles
Page 4: 20 profiles
Page 5: 20 profiles
14 matches
Bae Suzy https://mydramalist.com/people/422-suzy
Xiao Zhan https://mydramalist.com/people/13346-xiao-zhan
Cha Eun Woo https://mydramalist.com/people/10399-cha-eun-woo
Kim Se Jeong https://mydramalist.com/people/13855-kim-se-jung
Lim Yoon A https://mydramalist.com/people/972-im-yoon-ah
Lee Hye Ri https://mydramalist.com/people/2664-hyeri
Seo Kang Jun https://mydramalist.com/people/6214-seo-kang-joon
Wang Yi Bo https://mydramalist.com/people/13735-wang-yi-bo
Ji Soo https://mydramalist.com/people/12182-kim-ji-soo
V https://mydramalist.com/people/11489-v
Seo Hyun Jin https://mydramalist.com/people/941-seo-hyun-jin
Ju Jing Yi https://mydramalist.com/people/12464-ju-jing-yi
Krystal Jung https://mydramalist.com/people/1637-krystal
Cheng Xiao https://mydramalist.com/people/16297-cheng-xiao


In [15]:
#go into details section of each idol
#get number of dramas they're in 
DRAMA_ROWS_XPATH = (
    '//h5[@class="header"][normalize-space()="Drama"]'
    '/following-sibling::table[1]/tbody/tr'
)

num_dramas = []

for name, link in idol:
    driver.get(link)
    rows = driver.find_elements(By.XPATH, DRAMA_ROWS_XPATH)
    num_dramas.append(len(rows))
    print(name, len(rows))
    time.sleep(1)

results = list(zip([n for n, _ in idol], num_dramas))


Bae Suzy 18
Xiao Zhan 20
Cha Eun Woo 14
Kim Se Jeong 10
Lim Yoon A 15
Lee Hye Ri 18
Seo Kang Jun 22
Wang Yi Bo 13
Ji Soo 7
V 1
Seo Hyun Jin 25
Ju Jing Yi 20
Krystal Jung 16
Cheng Xiao 18


In [16]:
# Proportion of people who are idols
total_people = len(seen)  # every profile the scraper visited
proportion = len(idol) / total_people
print(f"Proportion of people who are idols: {proportion:.2%} ({len(idol)}/{total_people})")

# Average number of dramas
if num_dramas:
    average_dramas = sum(num_dramas) / len(num_dramas)
    print(f"Average number of dramas: {average_dramas:.1f}")
else:
    print("No dramas counted")

Proportion of people who are idols: 14.00% (14/100)
Average number of dramas: 15.5


In [4]:
driver.quit()